# Entity Divergence Graphs — Gephi Export

Two bipartite graphs, **all NER entities** as one side, parameterised by what sits
on the centre side:

- `centre_by = "metatopic"` — centre nodes are meta-topics (`topic_meta`).
  Only **news** and **talkshows** have `topic_meta`; **kamer is excluded** from this graph.
- `centre_by = "source"` — centre nodes are source sub-units (news outlet /
  talkshow programme / kamer doc type). All three arenas included.

Entities shared across many centre nodes sit central in Gephi's force layout;
entities unique to a single centre node sit peripheral — that's the point of
this graph, so **no edge-weight filtering** is applied.

**Cleaning is light:** stage-one normalisation + noise filter (drop empty/
single-char/purely-numeric tokens), alias merge via the `canonical` column of
the review workbook (if present) for `keep="yes"` rows, and a small junk
stop-list. Everything else — i.e. the entire long tail of one-off entities
never reviewed — is kept.

**Sources and columns used:**
| Arena | File | Sub-unit col | Has topic_meta |
|---|---|---|---|
| news | `news/analysis/df_with_NER.csv` | `outlet` | yes |
| talkshows | `subtitles/analysis/subs_with_NER.csv` | `program` | yes |
| kamer | `tweede_kamer/analysis/Tweede_Kamer_with_NER.csv` | `type` | no |

In [ ]:
import ast
import re
import pathlib
from collections import Counter
import pandas as pd

NB_DIR = pathlib.Path(".").resolve()

# ============================================================
# PARAMETERS
# ============================================================

SOURCES = {
    "news": {
        "path":          "../../news/analysis/df_with_NER.csv",
        "subunit_col":   "outlet",
        "node_type":     "news",
        "topic_col":     "topic",
        "meta_col":      "topic_meta",
        "has_topic_meta": True,
    },
    "talkshows": {
        "path":          "../../subtitles/analysis/subs_with_NER.csv",
        "subunit_col":   "program",
        "node_type":     "talkshows",
        "topic_col":     "topic",
        "meta_col":      "topic_meta",
        "has_topic_meta": True,
    },
    "kamer": {
        "path":          "../../tweede_kamer/analysis/Tweede_Kamer_with_NER.csv",
        "subunit_col":   "type",
        "node_type":     "kamer",
        "topic_col":     None,
        "meta_col":      None,
        "has_topic_meta": False,
    },
}

ENTITY_COLS = {
    "persons":   "PER",
    "orgs":      "ORG",
    "countries": "LOC",
}

REVIEW_XLSX   = "entity_review.xlsx"        # three-sheet workbook (PER/ORG/LOC); optional
JUNK_STOPLIST = {"wie", "dat.", "anders"}    # normalised lowercase; always dropped

MIN_EDGE_WEIGHT = 0.0   # explicit — do NOT drop low-weight edges in this graph

OUT_FILES = {
    "metatopic": ("metatopic_entity_nodes.csv", "metatopic_entity_edges.csv"),
    "source":    ("source_entity_nodes.csv",    "source_entity_edges.csv"),
}
# ============================================================

## 1. Helpers (stage-one parsing/normalisation)

In [ ]:
def parse_entity_list(cell):
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if s in ("", "[]", "nan"):
        return []
    try:
        result = ast.literal_eval(s)
        if isinstance(result, list):
            return [str(x) for x in result]
        return [str(result)]
    except (ValueError, SyntaxError):
        s = re.sub(r"^[\[\(]|[\]\)]$", "", s)
        return [x.strip().strip("'\"" ) for x in s.split(",") if x.strip()]


def normalise(s):
    return re.sub(r"\s+", " ", str(s).strip())


def slugify(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")


def make_unique_id(prefix, label, used_ids):
    """Slugify is lossy (e.g. 'M.' and 'M' both -> 'm') — disambiguate collisions
    with a numeric suffix so every node Id stays unique, as Gephi merges duplicates."""
    base = slugify(label) or "unk"
    candidate = f"{prefix}_{base}"
    if candidate not in used_ids:
        used_ids.add(candidate)
        return candidate
    i = 2
    while f"{candidate}_{i}" in used_ids:
        i += 1
    final = f"{candidate}_{i}"
    used_ids.add(final)
    return final

## 2. Build alias/drop lookup from review workbook (if present)

- `keep="no"` rows (any sheet) → dropped, added to `drop_set`
- `keep="yes"` rows → `entity_to_canonical[normalised raw] = canonical`
- `canonical_to_type_review[canonical] = sheet type` (the sheet a canonical
  appears on under `keep="yes"` already reflects any manual type correction)
- Entities never seen in the workbook pass through unmerged (canonical = themselves)
  and get their type from the actual data later.

In [ ]:
entity_to_canonical = {}
canonical_to_type_review = {}
drop_set = set(JUNK_STOPLIST)

review_path = (NB_DIR / REVIEW_XLSX).resolve()
if review_path.exists():
    for sheet in ["PER", "ORG", "LOC"]:
        sheet_df = pd.read_excel(review_path, sheet_name=sheet)
        for _, row in sheet_df.iterrows():
            key = normalise(row["entity"]).lower()
            if str(row["keep"]).strip().lower() == "no":
                drop_set.add(key)

    # second pass: keep=yes rows, skipping anything dropped above
    for sheet in ["PER", "ORG", "LOC"]:
        sheet_df = pd.read_excel(review_path, sheet_name=sheet)
        kept = sheet_df[sheet_df["keep"].astype(str).str.strip().str.lower() == "yes"]
        for _, row in kept.iterrows():
            key = normalise(row["entity"]).lower()
            if key in drop_set:
                continue
            canonical = normalise(row["canonical"])
            entity_to_canonical[key] = canonical
            canonical_to_type_review[canonical] = sheet

    print(f"Review workbook found: {len(entity_to_canonical)} alias mappings, "
          f"{len(drop_set)} dropped keys (incl. junk stop-list)")
else:
    print("No review workbook found — all entities pass through unmerged, "
          "only junk stop-list applied.")

## 3. Load source files

In [ ]:
frames = {}

for arena, cfg in SOURCES.items():
    abs_path = (NB_DIR / cfg["path"]).resolve()
    if not abs_path.exists():
        print(f"[SKIP] {arena}: not found")
        continue
    df = pd.read_csv(abs_path)
    df[cfg["subunit_col"]] = df[cfg["subunit_col"]].fillna("UNKNOWN").astype(str).str.strip()
    if cfg["has_topic_meta"]:
        df = df[df[cfg["topic_col"]] != -1].copy()
        df[cfg["meta_col"]] = df[cfg["meta_col"]].fillna("UNKNOWN").astype(str).str.strip()
    frames[arena] = df
    print(f"[OK] {arena}: {len(df):,} docs, sub-unit='{cfg['subunit_col']}', "
          f"topic_meta={'yes' if cfg['has_topic_meta'] else 'no'}")

## 4. Parse all documents once → shared long-format table

One row per (arena, subunit, topic_meta, doc_id, canonical), deduplicated
within each document. `raw_type` (PER/ORG/LOC, from the originating column) is
kept per mention so un-reviewed canonicals can get a type by majority vote.

In [ ]:
records = []
noise_dropped = 0
junk_dropped  = 0

for arena, cfg in SOURCES.items():
    if arena not in frames:
        continue
    df          = frames[arena]
    subunit_col = cfg["subunit_col"]
    meta_col    = cfg["meta_col"]

    for row_idx, row in df.iterrows():
        subunit    = row[subunit_col]
        topic_meta = row[meta_col] if cfg["has_topic_meta"] else None
        doc_id     = f"{arena}:{row_idx}"

        # dedupe (canonical, raw_type) within this document
        doc_seen = {}   # canonical -> raw_type (first seen)
        for col, raw_type in ENTITY_COLS.items():
            for raw in parse_entity_list(row.get(col)):
                norm = normalise(raw)
                if len(norm) <= 1 or norm.isdigit():
                    noise_dropped += 1
                    continue
                key = norm.lower()
                if key in drop_set:
                    junk_dropped += 1
                    continue
                canonical = entity_to_canonical.get(key, norm)
                if canonical not in doc_seen:
                    doc_seen[canonical] = raw_type

        for canonical, raw_type in doc_seen.items():
            records.append({
                "arena":      arena,
                "subunit":    subunit,
                "topic_meta": topic_meta,
                "doc_id":     doc_id,
                "canonical":  canonical,
                "raw_type":   raw_type,
            })

long_df = pd.DataFrame(records)
print(f"(doc, canonical) pairings: {len(long_df):,}")
print(f"Noise-dropped mentions (empty/single-char/numeric): {noise_dropped:,}")
print(f"Junk/keep=no-dropped mentions: {junk_dropped:,}")
print(f"Distinct canonical entities: {long_df['canonical'].nunique():,}")

## 5. Resolve entity_type for every canonical

Review-corrected type wins where available; otherwise the dominant raw type
(doc-level majority vote across persons/orgs/countries) is used.

In [ ]:
type_doc = long_df[["canonical", "doc_id", "raw_type"]].drop_duplicates()
type_counts = (
    type_doc.groupby(["canonical", "raw_type"])
    .size()
    .reset_index(name="n")
)
computed_dominant_type = (
    type_counts
    .loc[type_counts.groupby("canonical")["n"].idxmax()]
    .set_index("canonical")["raw_type"]
)

def resolve_type(canonical):
    return canonical_to_type_review.get(canonical, computed_dominant_type.get(canonical, ""))

canonical_type = {c: resolve_type(c) for c in long_df["canonical"].unique()}
print(f"Resolved types for {len(canonical_type):,} canonicals "
      f"({sum(1 for c in canonical_type if c in canonical_to_type_review)} from review)")

## 6. Build graph function

In [ ]:
def build_divergence_graph(centre_by):
    used_ids = set()

    if centre_by == "metatopic":
        scope = long_df[long_df["arena"].isin(
            [a for a, cfg in SOURCES.items() if cfg["has_topic_meta"]]
        )].copy()
        scope["centre_label"] = scope["topic_meta"]
        scope["centre_node_type"] = "metatopic"

        # centre totals: doc counts per topic_meta across in-scope arenas
        centre_doc_totals = {}   # label -> total docs
        for arena, cfg in SOURCES.items():
            if not cfg["has_topic_meta"] or arena not in frames:
                continue
            for meta, grp in frames[arena].groupby(cfg["meta_col"]):
                centre_doc_totals[meta] = centre_doc_totals.get(meta, 0) + len(grp)
        centre_rows = [
            {
                "Id":          make_unique_id("meta", label, used_ids),
                "Label":       label,
                "node_type":   "metatopic",
                "entity_type": "",
                "size":        total,
            }
            for label, total in centre_doc_totals.items()
        ]
        centre_id = {r["Label"]: r["Id"] for r in centre_rows}

    elif centre_by == "source":
        scope = long_df.copy()
        scope["centre_label"] = scope["subunit"]
        scope["centre_node_type"] = scope["arena"]

        centre_doc_totals = {}   # label -> total docs
        subunit_node_type = {}   # label -> arena node_type
        for arena, cfg in SOURCES.items():
            if arena not in frames:
                continue
            for subunit, grp in frames[arena].groupby(cfg["subunit_col"]):
                centre_doc_totals[subunit] = len(grp)
                subunit_node_type[subunit] = cfg["node_type"]
        centre_rows = [
            {
                "Id":          make_unique_id("src", label, used_ids),
                "Label":       label,
                "node_type":   subunit_node_type[label],
                "entity_type": "",
                "size":        total,
            }
            for label, total in centre_doc_totals.items()
        ]
        centre_id = {r["Label"]: r["Id"] for r in centre_rows}

    else:
        raise ValueError(f"Unknown centre_by: {centre_by}")

    # --- entity nodes (scoped to documents included in this graph) ---
    entity_doc_freq = scope.groupby("canonical")["doc_id"].nunique()
    entity_rows = [
        {
            "Id":          make_unique_id("ent", canonical, used_ids),
            "Label":       canonical,
            "node_type":   "entity",
            "entity_type": canonical_type.get(canonical, ""),
            "size":        int(freq),
        }
        for canonical, freq in entity_doc_freq.items()
    ]
    entity_id = {r["Label"]: r["Id"] for r in entity_rows}

    nodes_df = pd.DataFrame(centre_rows + entity_rows)
    dupes = nodes_df[nodes_df.duplicated("Id", keep=False)]
    if len(dupes):
        print(f"  WARNING ({centre_by}): duplicate node Ids: {dupes['Id'].tolist()[:10]}")

    # --- edges: coverage share, no min-weight filtering ---
    mention_counts = (
        scope.groupby(["centre_label", "canonical"])["doc_id"]
        .nunique()
        .reset_index(name="n")
    )

    edge_rows = []
    for _, row in mention_counts.iterrows():
        label = row["centre_label"]
        total = centre_doc_totals.get(label)
        if not total:
            continue
        src = centre_id.get(label)
        tgt = entity_id.get(row["canonical"])
        if src is None or tgt is None:
            continue
        weight = row["n"] / total
        if weight < MIN_EDGE_WEIGHT:
            continue
        edge_rows.append({
            "Source": src,
            "Target": tgt,
            "Weight": round(weight, 6),
            "Type":   "Undirected",
        })

    edges_df = pd.DataFrame(edge_rows)

    return nodes_df, edges_df, centre_rows, entity_rows

## 7. Run both graphs and export

In [ ]:
print("=" * 60)

for centre_by in ["metatopic", "source"]:
    nodes_df, edges_df, centre_rows, entity_rows = build_divergence_graph(centre_by)

    nodes_file, edges_file = OUT_FILES[centre_by]
    nodes_path = NB_DIR / nodes_file
    edges_path = NB_DIR / edges_file
    nodes_df.to_csv(nodes_path, index=False)
    edges_df.to_csv(edges_path, index=False)

    # entity degree = number of distinct centre nodes each entity connects to
    entity_degree = edges_df.groupby("Target")["Source"].nunique()

    print(f"centre_by = {centre_by!r}")
    print(f"  Centre nodes : {len(centre_rows)}")
    print(f"  Entity nodes : {len(entity_rows)}")
    print(f"  Total nodes  : {len(nodes_df)}")
    print(f"  Total edges  : {len(edges_df)}")
    if len(entity_degree):
        print(f"  Entity degree: median={entity_degree.median():.1f}  max={entity_degree.max()}")
        top_central = entity_degree.sort_values(ascending=False).head(5)
        id_to_label = dict(zip(nodes_df["Id"], nodes_df["Label"]))
        print("  Most central entities:")
        for ent_id, deg in top_central.items():
            print(f"      {id_to_label.get(ent_id, ent_id):<25s} degree={deg}")
    print(f"  -> {nodes_file}, {edges_file}")
    print()

print("=" * 60)